<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/assignments/conditional_expectation_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Q1) Prediction of fraudulent transactions

A bank monitors credit-card transactions for possible fraud.

For each transaction, let the sample space be

$$
\Omega=\{\omega_1,\omega_2,\omega_3,\omega_4\},
$$

where:

* $\omega_1$: low-risk customer, legitimate transaction,
* $\omega_2$: low-risk customer, fraudulent transaction,
* $\omega_3$: high-risk customer, legitimate transaction,
* $\omega_4$: high-risk customer, fraudulent transaction.

Assume the probabilities are

$$
P(\{\omega_1\})=0.50,\qquad
P(\{\omega_2\})=0.10,\qquad
P(\{\omega_3\})=0.20,\qquad
P(\{\omega_4\})=0.20.
$$

Let $X:\Omega\to\mathbb R$ denote the fraud indicator:

$$
X(\omega_1)=0,\qquad
X(\omega_2)=1,\qquad
X(\omega_3)=0,\qquad
X(\omega_4)=1.
$$

Thus $X=1$ means the transaction is fraudulent, and $X=0$ means it is legitimate.

Now suppose the bank does not observe the full outcome $\omega$, but only the customer's risk class. Define $Y:\Omega\to\mathbb R$ by

$$
Y(\omega_1)=0,\qquad
Y(\omega_2)=0,\qquad
Y(\omega_3)=1,\qquad
Y(\omega_4)=1.
$$

Thus:

* $Y=0$ means “low-risk customer,”
* $Y=1$ means “high-risk customer.”

Answer the following.

1. Write down the probability space $(\Omega,\mathfrak F,P)$, where $\mathfrak F=\mathfrak P(\Omega)$.

2. Compute the $\sigma$-algebra generated by $Y$, namely $\sigma(Y)$.

3. Explain why $E[X\mid Y]$ must be $\sigma(Y)$-measurable, and therefore must be constant on the sets $\{ \omega_1,\omega_2\}$ and $\{\omega_3,\omega_4\}$.

4. Compute $E[X\mid Y]$ explicitly and show that

$$
E[X\mid Y]= \frac{1}{6} \mathbb{1}_{\{Y=0\}} + \frac{1}{2} \mathbb{1}_{\{Y=1\}}
$$

5. Verify the defining property of conditional expectation by checking that for every set $A\in\sigma(Y)$,

$$
\int_A E[X\mid Y]\,dP=\int_A X\,dP.
$$

6. Find the Conditional PMF $p_{X|Y}(x \mid y)$.

7. Verify the identity:
$$\mathbb{E}[X\mid Y=y] = \sum_{x \in \{0,1\}} x \cdot p_{X|Y}(x \mid y)$$.

8. Calculate the **Mean Square Error (MSE)**

9. How would the **Mean Square Error (MSE)** change if the bank decided to ignore the customer risk class $Y$ and instead used the global average fraud rate ($E[X] = 0.30$) as a constant estimate for every transaction? Calculate this error and explain its relationship to the **Law of Total Variance**.

10. Using the joint distribution of $(X, Y)$ derived from the sample space:

    a) Compute the entropy $H(X)$, representing the bank's initial uncertainty regarding fraud.

    b) Compute the conditional entropy $H(X \mid Y)$, representing the uncertainty remaining after the customer's risk class is revealed.

    c) Calculate the Mutual Information $I(X; Y)$.

    d) Based on these values, what percentage of the total uncertainty about fraud is removed by knowing the customer's risk class?

In [1]:
import numpy as np
from math import log2

# Define the sample space and probabilities
omega = ['omega1', 'omega2', 'omega3', 'omega4']
P = {
    'omega1': 0.50,
    'omega2': 0.10,
    'omega3': 0.20,
    'omega4': 0.20
}

# Define random variable X (fraud indicator)
X_map = {
    'omega1': 0,
    'omega2': 1,
    'omega3': 0,
    'omega4': 1
}

# Define random variable Y (customer risk class)
Y_map = {
    'omega1': 0,
    'omega2': 0,
    'omega3': 1,
    'omega4': 1
}

# --- 1. Write down the probability space (Omega, F, P) ---
print("--- 1. Probability Space ---")
print(f"Omega (sample space): {omega}")
print("F (sigma-algebra): F = P(Omega) = the power set of Omega (all 2^4 = 16 subsets).")
print("P (probability measure): P({\omega_1})=0.50, P({\omega_2})=0.10, P({\omega_3})=0.20, P({\omega_4})=0.20.")
print("\n")

# --- 2. Compute the sigma-algebra generated by Y, namely sigma(Y) ---
print("--- 2. Sigma-algebra generated by Y, sigma(Y) ---")
# Y=0 for omega1, omega2
# Y=1 for omega3, omega4

Y_inv_0 = {'omega1', 'omega2'}
Y_inv_1 = {'omega3', 'omega4'}

sigma_Y = [
    set(), # Empty set
    Y_inv_0, # {Y=0}
    Y_inv_1, # {Y=1}
    Y_inv_0.union(Y_inv_1) # Omega
]
print(f"The preimages are Y^-1(0) = {Y_inv_0} and Y^-1(1) = {Y_inv_1}.")
print(f"sigma(Y) = {{ \u2205, {Y_inv_0}, {Y_inv_1}, \u03A9 }}")
print("\n")

# --- 3. Explain why E[X|Y] must be sigma(Y)-measurable ---
print("--- 3. Explanation for E[X|Y] being sigma(Y)-measurable ---")
print("The conditional expectation E[X|Y] is, by definition, a random variable that is sigma(Y)-measurable.")
print("This means that E[X|Y] must be constant on the sets generated by Y (the atoms of sigma(Y)).")
print("Specifically, E[X|Y] must be constant on Y^-1(0) = {\u03c9_1, \u03c9_2} and on Y^-1(1) = {\u03c9_3, \u03c9_4}.")
print("\n")

# --- 4. Compute E[X|Y] explicitly ---
print("--- 4. Compute E[X|Y] explicitly ---")

P_Y0 = P['omega1'] + P['omega2'] # P(Y=0)
P_Y1 = P['omega3'] + P['omega4'] # P(Y=1)

# E[X | Y=0] = P(X=1, Y=0) / P(Y=0)
P_X1Y0 = P['omega2'] # X=1, Y=0
E_X_given_Y0 = P_X1Y0 / P_Y0

# E[X | Y=1] = P(X=1, Y=1) / P(Y=1)
P_X1Y1 = P['omega4'] # X=1, Y=1
E_X_given_Y1 = P_X1Y1 / P_Y1

print(f"P(Y=0) = P(\u03c9_1) + P(\u03c9_2) = {P['omega1']} + {P['omega2']} = {P_Y0:.2f}")
print(f"P(Y=1) = P(\u03c9_3) + P(\u03c9_4) = {P['omega3']} + {P['omega4']} = {P_Y1:.2f}")

print(f"E[X | Y=0] = P(X=1, Y=0) / P(Y=0) = P(\u03c9_2) / P(Y=0) = {P['omega2']:.2f} / {P_Y0:.2f} = {E_X_given_Y0:.2f} (which is 1/6)")
print(f"E[X | Y=1] = P(X=1, Y=1) / P(Y=1) = P(\u03c9_4) / P(Y=1) = {P['omega4']:.2f} / {P_Y1:.2f} = {E_X_given_Y1:.2f} (which is 1/2)")

print(f"Thus, E[X|Y] = {E_X_given_Y0:.2f} * \u2207_{{Y=0}} + {E_X_given_Y1:.2f} * \u2207_{{Y=1}}")
print(f"Which matches the given form: E[X|Y] = (1/6) * \u2207_{{Y=0}} + (1/2) * \u2207_{{Y=1}}")
print("\n")

# --- 5. Verify the defining property of conditional expectation ---
print("--- 5. Verify the defining property of conditional expectation ---")

# For A = {Y=0}
# LHS: Integral over A of E[X|Y] dP = E[X|Y=0] * P(Y=0)
LHS_Y0 = E_X_given_Y0 * P_Y0
# RHS: Integral over A of X dP = E[X * I(Y=0)]
RHS_Y0 = X_map['omega1'] * P['omega1'] + X_map['omega2'] * P['omega2']

print(f"For A = {{Y=0}}:")
print(f"  \u222B_A E[X|Y] dP = E[X|Y=0] * P(Y=0) = {E_X_given_Y0:.2f} * {P_Y0:.2f} = {LHS_Y0:.2f}")
print(f"  \u222B_A X dP = X(\u03c9_1)P(\u03c9_1) + X(\u03c9_2)P(\u03c9_2) = {X_map['omega1']}*{P['omega1']} + {X_map['omega2']}*{P['omega2']} = {RHS_Y0:.2f}")
print(f"  LHS ({LHS_Y0:.2f}) == RHS ({RHS_Y0:.2f}) -> {'Verified' if np.isclose(LHS_Y0, RHS_Y0) else 'Failed'}")

# For A = {Y=1}
# LHS: Integral over A of E[X|Y] dP = E[X|Y=1] * P(Y=1)
LHS_Y1 = E_X_given_Y1 * P_Y1
# RHS: Integral over A of X dP = E[X * I(Y=1)]
RHS_Y1 = X_map['omega3'] * P['omega3'] + X_map['omega4'] * P['omega4']

print(f"For A = {{Y=1}}:")
print(f"  \u222B_A E[X|Y] dP = E[X|Y=1] * P(Y=1) = {E_X_given_Y1:.2f} * {P_Y1:.2f} = {LHS_Y1:.2f}")
print(f"  \u222B_A X dP = X(\u03c9_3)P(\u03c9_3) + X(\u03c9_4)P(\u03c9_4) = {X_map['omega3']}*{P['omega3']} + {X_map['omega4']}*{P['omega4']} = {RHS_Y1:.2f}")
print(f"  LHS ({LHS_Y1:.2f}) == RHS ({RHS_Y1:.2f}) -> {'Verified' if np.isclose(LHS_Y1, RHS_Y1) else 'Failed'}")
print("\n")

# --- 6. Find the Conditional PMF p_X|Y(x | y) ---
print("--- 6. Conditional PMF p_X|Y(x | y) ---")

# P(X=0, Y=0) = P(omega1) = 0.50
# P(X=1, Y=0) = P(omega2) = 0.10
# P(X=0, Y=1) = P(omega3) = 0.20
# P(X=1, Y=1) = P(omega4) = 0.20

pmf_X_given_Y = {}

pmf_X_given_Y[(0,0)] = P['omega1'] / P_Y0 # P(X=0|Y=0)
pmf_X_given_Y[(1,0)] = P['omega2'] / P_Y0 # P(X=1|Y=0)
pmf_X_given_Y[(0,1)] = P['omega3'] / P_Y1 # P(X=0|Y=1)
pmf_X_given_Y[(1,1)] = P['omega4'] / P_Y1 # P(X=1|Y=1)

print(f"p_X|Y(0 | 0) = P(X=0, Y=0) / P(Y=0) = {P['omega1']:.2f} / {P_Y0:.2f} = {pmf_X_given_Y[(0,0)]:.2f}")
print(f"p_X|Y(1 | 0) = P(X=1, Y=0) / P(Y=0) = {P['omega2']:.2f} / {P_Y0:.2f} = {pmf_X_given_Y[(1,0)]:.2f}")
print(f"p_X|Y(0 | 1) = P(X=0, Y=1) / P(Y=1) = {P['omega3']:.2f} / {P_Y1:.2f} = {pmf_X_given_Y[(0,1)]:.2f}")
print(f"p_X|Y(1 | 1) = P(X=1, Y=1) / P(Y=1) = {P['omega4']:.2f} / {P_Y1:.2f} = {pmf_X_given_Y[(1,1)]:.2f}")
print("\n")

# --- 7. Verify the identity: E[X|Y=y] = sum(x * p_X|Y(x | y)) ---
print("--- 7. Verify identity: E[X|Y=y] = \u2211 x * p_X|Y(x | y) ---")

# For y=0
sum_X_pmf_Y0 = 0 * pmf_X_given_Y[(0,0)] + 1 * pmf_X_given_Y[(1,0)]
print(f"For y=0: \u2211 x * p_X|Y(x | 0) = 0 * {pmf_X_given_Y[(0,0)]:.2f} + 1 * {pmf_X_given_Y[(1,0)]:.2f} = {sum_X_pmf_Y0:.2f}")
print(f"  E[X|Y=0] = {E_X_given_Y0:.2f}. Identity verified: {'True' if np.isclose(sum_X_pmf_Y0, E_X_given_Y0) else 'False'}")

# For y=1
sum_X_pmf_Y1 = 0 * pmf_X_given_Y[(0,1)] + 1 * pmf_X_given_Y[(1,1)]
print(f"For y=1: \u2211 x * p_X|Y(x | 1) = 0 * {pmf_X_given_Y[(0,1)]:.2f} + 1 * {pmf_X_given_Y[(1,1)]:.2f} = {sum_X_pmf_Y1:.2f}")
print(f"  E[X|Y=1] = {E_X_given_Y1:.2f}. Identity verified: {'True' if np.isclose(sum_X_pmf_Y1, E_X_given_Y1) else 'False'}")
print("\n")

# --- 8. Calculate the Mean Square Error (MSE) ---
print("--- 8. Calculate Mean Square Error (MSE) ---")

# E[X|Y] takes value E_X_given_Y0 for omega1, omega2
# E[X|Y] takes value E_X_given_Y1 for omega3, omega4

mse_sum = 0
mse_sum += (X_map['omega1'] - E_X_given_Y0)**2 * P['omega1']
mse_sum += (X_map['omega2'] - E_X_given_Y0)**2 * P['omega2']
mse_sum += (X_map['omega3'] - E_X_given_Y1)**2 * P['omega3']
mse_sum += (X_map['omega4'] - E_X_given_Y1)**2 * P['omega4']

print(f"MSE = E[(X - E[X|Y])^2]")
print(f"    = (X(\u03c9_1) - E[X|Y=0])^2 P(\u03c9_1) + ...")
print(f"    = ({0} - {E_X_given_Y0:.2f})^2 * {P['omega1']:.2f} + \
         ({1} - {E_X_given_Y0:.2f})^2 * {P['omega2']:.2f} + \
         ({0} - {E_X_given_Y1:.2f})^2 * {P['omega3']:.2f} + \
         ({1} - {E_X_given_Y1:.2f})^2 * {P['omega4']:.2f}")
print(f"    = {mse_sum:.4f}")
print("\n")

# --- 9. MSE with global average fraud rate E[X] = 0.30 ---
print("--- 9. MSE with global average fraud rate E[X] = 0.30 ---")

E_X = sum(X_map[o] * P[o] for o in omega)
print(f"E[X] (global average fraud rate) = {E_X:.2f}")

mse_naive = 0
mse_naive += (X_map['omega1'] - E_X)**2 * P['omega1']
mse_naive += (X_map['omega2'] - E_X)**2 * P['omega2']
mse_naive += (X_map['omega3'] - E_X)**2 * P['omega3']
mse_naive += (X_map['omega4'] - E_X)**2 * P['omega4']

# This is simply Var(X)
Var_X = E_X - (E_X)**2 # This is wrong for Bernoulli, Var(X) = E[X^2] - (E[X])^2
E_X2 = sum(X_map[o]**2 * P[o] for o in omega)
Var_X = E_X2 - (E_X)**2

print(f"MSE if using E[X] = {E_X:.2f} as estimate = E[(X - E[X])^2] = Var(X)")
print(f"    = ({0} - {E_X:.2f})^2 * {P['omega1']:.2f} + \
         ({1} - {E_X:.2f})^2 * {P['omega2']:.2f} + \
         ({0} - {E_X:.2f})^2 * {P['omega3']:.2f} + \
         ({1} - {E_X:.2f})^2 * {P['omega4']:.2f}")
print(f"    = {mse_naive:.4f}")
print(f"    (Also calculated as Var(X) = E[X^2] - (E[X])^2 = {E_X2:.2f} - ({E_X:.2f})^2 = {Var_X:.4f})")

print(f"\nComparing the two MSE values:")
print(f"MSE with E[X|Y] (knowing risk class): {mse_sum:.4f}")
print(f"MSE with E[X] (ignoring risk class): {mse_naive:.4f}")
print(f"The MSE is reduced from {mse_naive:.4f} to {mse_sum:.4f} by incorporating the customer risk class.")

print("\nRelationship to the Law of Total Variance:")
print("The Law of Total Variance states: Var(X) = E[Var(X|Y)] + Var(E[X|Y])")
print("Here, Var(X) is the MSE when ignoring Y (the naive MSE).")
print("E[Var(X|Y)] is the MSE when using E[X|Y] (the conditional MSE).")
print("Var(E[X|Y]) is the variance explained by Y (the reduction in variance due to knowing Y).")

# Calculate E[Var(X|Y)]
Var_X_given_Y0 = E_X_given_Y0 - (E_X_given_Y0)**2 # For Bernoulli, Var(X|Y=y) = E[X|Y=y](1-E[X|Y=y])
# P(X=1|Y=0) is E[X|Y=0]
Var_X_given_Y0 = (P['omega2']/P_Y0) * (1 - P['omega2']/P_Y0)
Var_X_given_Y1 = (P['omega4']/P_Y1) * (1 - P['omega4']/P_Y1)

E_Var_X_given_Y = Var_X_given_Y0 * P_Y0 + Var_X_given_Y1 * P_Y1

print(f"E[Var(X|Y)] = Var(X|Y=0)P(Y=0) + Var(X|Y=1)P(Y=1) = {Var_X_given_Y0:.4f}*{P_Y0:.2f} + {Var_X_given_Y1:.4f}*{P_Y1:.2f} = {E_Var_X_given_Y:.4f}")

# Calculate Var(E[X|Y])
# E[X|Y] is a random variable that takes values E_X_given_Y0 with prob P_Y0 and E_X_given_Y1 with prob P_Y1
E_E_X_given_Y = E_X_given_Y0 * P_Y0 + E_X_given_Y1 * P_Y1 # This should be E[X]
Var_E_X_given_Y = (E_X_given_Y0 - E_X)**2 * P_Y0 + (E_X_given_Y1 - E_X)**2 * P_Y1

print(f"Var(E[X|Y]) = (E[X|Y=0] - E[X])^2 P(Y=0) + (E[X|Y=1] - E[X])^2 P(Y=1) = {Var_E_X_given_Y:.4f}")

print(f"\nVerify Law of Total Variance: Var(X) = E[Var(X|Y)] + Var(E[X|Y])")
print(f"LHS: Var(X) = {Var_X:.4f}")
print(f"RHS: {E_Var_X_given_Y:.4f} + {Var_E_X_given_Y:.4f} = {E_Var_X_given_Y + Var_E_X_given_Y:.4f}")
print(f"Verification: {'True' if np.isclose(Var_X, E_Var_X_given_Y + Var_E_X_given_Y) else 'False'}")

print("The reduction in MSE from ignoring Y to using E[X|Y] is exactly Var(E[X|Y]).")
print(f"Reduction = {mse_naive - mse_sum:.4f}. Var(E[X|Y]) = {Var_E_X_given_Y:.4f}. {'Verified' if np.isclose(mse_naive - mse_sum, Var_E_X_given_Y) else 'Failed'}")
print("\n")

# --- 10. Information Theory ---
print("--- 10. Information Theory ---")

# Calculate P(X=0) and P(X=1)
P_X0 = P['omega1'] + P['omega3']
P_X1 = P['omega2'] + P['omega4']
print(f"P(X=0) = {P_X0:.2f}, P(X=1) = {P_X1:.2f}")
print(f"P(Y=0) = {P_Y0:.2f}, P(Y=1) = {P_Y1:.2f}")

# a) Compute H(X)
def entropy(p_dist):
    h = 0
    for p in p_dist:
        if p > 0:
            h -= p * log2(p)
    return h

H_X = entropy([P_X0, P_X1])
print(f"\na) H(X) = -({P_X0:.2f} log2({P_X0:.2f}) + {P_X1:.2f} log2({P_X1:.2f})) = {H_X:.4f} bits")

# b) Compute H(X|Y)
# H(X|Y=0)
px_y0 = [pmf_X_given_Y[(0,0)], pmf_X_given_Y[(1,0)]]
H_X_given_Y0 = entropy(px_y0)
print(f"  H(X|Y=0) = -({px_y0[0]:.2f} log2({px_y0[0]:.2f}) + {px_y0[1]:.2f} log2({px_y0[1]:.2f})) = {H_X_given_Y0:.4f} bits")

# H(X|Y=1)
px_y1 = [pmf_X_given_Y[(0,1)], pmf_X_given_Y[(1,1)]]
H_X_given_Y1 = entropy(px_y1)
print(f"  H(X|Y=1) = -({px_y1[0]:.2f} log2({px_y1[0]:.2f}) + {px_y1[1]:.2f} log2({px_y1[1]:.2f})) = {H_X_given_Y1:.4f} bits")

H_X_given_Y = P_Y0 * H_X_given_Y0 + P_Y1 * H_X_given_Y1
print(f"b) H(X|Y) = P(Y=0)H(X|Y=0) + P(Y=1)H(X|Y=1) = {P_Y0:.2f} * {H_X_given_Y0:.4f} + {P_Y1:.2f} * {H_X_given_Y1:.4f} = {H_X_given_Y:.4f} bits")

# c) Calculate Mutual Information I(X;Y)
I_X_Y = H_X - H_X_given_Y
print(f"c) I(X;Y) = H(X) - H(X|Y) = {H_X:.4f} - {H_X_given_Y:.4f} = {I_X_Y:.4f} bits")

# d) Percentage of uncertainty removed
percentage_removed = (I_X_Y / H_X) * 100
print(f"d) Percentage of total uncertainty about fraud removed by knowing customer risk class:")
print(f"   (I(X;Y) / H(X)) * 100 = ({I_X_Y:.4f} / {H_X:.4f}) * 100 = {percentage_removed:.2f}%")


--- 1. Probability Space ---
Omega (sample space): ['omega1', 'omega2', 'omega3', 'omega4']
F (sigma-algebra): F = P(Omega) = the power set of Omega (all 2^4 = 16 subsets).
P (probability measure): P({\omega_1})=0.50, P({\omega_2})=0.10, P({\omega_3})=0.20, P({\omega_4})=0.20.


--- 2. Sigma-algebra generated by Y, sigma(Y) ---
The preimages are Y^-1(0) = {'omega1', 'omega2'} and Y^-1(1) = {'omega4', 'omega3'}.
sigma(Y) = { ∅, {'omega1', 'omega2'}, {'omega4', 'omega3'}, Ω }


--- 3. Explanation for E[X|Y] being sigma(Y)-measurable ---
The conditional expectation E[X|Y] is, by definition, a random variable that is sigma(Y)-measurable.
This means that E[X|Y] must be constant on the sets generated by Y (the atoms of sigma(Y)).
Specifically, E[X|Y] must be constant on Y^-1(0) = {ω_1, ω_2} and on Y^-1(1) = {ω_3, ω_4}.


--- 4. Compute E[X|Y] explicitly ---
P(Y=0) = P(ω_1) + P(ω_2) = 0.5 + 0.1 = 0.60
P(Y=1) = P(ω_3) + P(ω_4) = 0.2 + 0.2 = 0.40
E[X | Y=0] = P(X=1, Y=0) / P(Y=0) = P(ω_2) / P(Y

<>:33: SyntaxWarning: invalid escape sequence '\o'
<>:33: SyntaxWarning: invalid escape sequence '\o'
/tmp/ipykernel_4855/2352569818.py:33: SyntaxWarning: invalid escape sequence '\o'
  print("P (probability measure): P({\omega_1})=0.50, P({\omega_2})=0.10, P({\omega_3})=0.20, P({\omega_4})=0.20.")


# Q2) Conditional Density of the Multivariate Gaussian

Let $\mathbf{Z}$ be the concatenated random matrix
$\mathbf{Z} = \begin{bmatrix} \mathbf{X} \\ \mathbf{Y} \end{bmatrix} \in \mathbb{R}^{n+m}$.


The random variable $\mathbf{Z}$ is said to have a **joint multivariate normal distribution** if every linear combination of its components follows a univariate normal distribution.

For $\mathbf{Z}$ to possess a **joint density function** (meaning its distribution is absolutely continuous with respect to the Lebesgue measure on $\mathbb{R}^{n+m}$), its covariance matrix must be non-singular (positive definite).


The joint density function $f_{\mathbf{X}, \mathbf{Y}}(\mathbf{z})$ for $\mathbf{z} \in \mathbb{R}^{n+m}$ is given by:

$$f_{\mathbf{X}, \mathbf{Y}}(\mathbf{z}) = \frac{1}{\sqrt{(2\pi)^{n+m} |\boldsymbol{\Sigma}|}} \exp\left( -\frac{1}{2} (\mathbf{z} - \boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{z} - \boldsymbol{\mu}) \right).$$

Here:

* **Mean ($\boldsymbol{\mu}$):** $\boldsymbol{\mu} = E[\mathbf{Z}] = \begin{bmatrix} \boldsymbol{\mu}_X \\ \boldsymbol{\mu}_Y \end{bmatrix} \in \mathbb{R}^{n+m}$, where $\boldsymbol{\mu}_X \in \mathbb{R}^n$ and $\boldsymbol{\mu}_Y \in \mathbb{R}^m$.
* **Covariance Matrix ($\boldsymbol{\Sigma}$):** A symmetric, positive definite matrix $\boldsymbol{\Sigma} \in \mathbb{R}^{(n+m) \times (n+m)}$ partitioned as:
$$\boldsymbol{\Sigma} = \begin{bmatrix} \boldsymbol{\Sigma}_{XX} & \boldsymbol{\Sigma}_{XY} \\ \boldsymbol{\Sigma}_{YX} & \boldsymbol{\Sigma}_{YY} \end{bmatrix}$$
    * $\boldsymbol{\Sigma}_{XX} = \text{Var}(\mathbf{X})$ (size $n \times n$)
    * $\boldsymbol{\Sigma}_{YY} = \text{Var}(\mathbf{Y})$ (size $m \times m$)
    * $\boldsymbol{\Sigma}_{XY} = \text{Cov}(\mathbf{X}, \mathbf{Y}) = \boldsymbol{\Sigma}_{YX}^T$ (size $n \times m$)
* **Determinant ($|\boldsymbol{\Sigma}|$):** The normalization factor ensuring the total integral is 1.

1. Show that the conditional distribution is given by $f_{\mathbf{X}|\mathbf{Y}}(\mathbf{x} \mid \mathbf{y})$

$$f_{\mathbf{X}|\mathbf{Y}}(\mathbf{x} \mid \mathbf{y}) = \frac{1}{\sqrt{(2\pi)^n |\mathbf{S}_{XX}|}} \exp\left( -\frac{1}{2} (\mathbf{x} - \boldsymbol{\mu}^*)^T \boldsymbol{\Sigma}_{X\mid Y}^{-1} (\mathbf{x} - \boldsymbol{\mu}^*) \right),$$
where
* $\boldsymbol{\Sigma}_{X\mid Y} = \boldsymbol{\Sigma}_{XX} - \boldsymbol{\Sigma}_{XY}\boldsymbol{\Sigma}_{YY}^{-1}\boldsymbol{\Sigma}_{YX}$
* $\boldsymbol{\mu}^* = \boldsymbol{\mu}_x + \boldsymbol{\Sigma}_{XY}\boldsymbol{\Sigma}_{YY}^{-1}(\mathbf{y} - \boldsymbol{\mu}_y)$

In [4]:
print("""To show that the conditional distribution $f_{\mathbf{X}|\mathbf{Y}}(\mathbf{x} \mid \mathbf{y})$ for a jointly Gaussian random vector $\mathbf{Z} = \begin{bmatrix} \mathbf{X} \\ \mathbf{Y} \end{bmatrix}$ is also Gaussian, we start with the definition of conditional probability density:

$$f_{\mathbf{X}|\mathbf{Y}}(\mathbf{x} \mid \mathbf{y}) = \frac{f_{\mathbf{X}, \mathbf{Y}}(\mathbf{x}, \mathbf{y})}{f_{\mathbf{Y}}(\mathbf{y})}$$

We know the joint density $f_{\mathbf{X}, \mathbf{Y}}(\mathbf{z})$ and the marginal density $f_{\mathbf{Y}}(\mathbf{y})$ (which is also Gaussian). The key is to complete the square in the exponent of the joint density function.

Let $\mathbf{Z} = \begin{bmatrix} \mathbf{X} \\ \mathbf{Y} \end{bmatrix}$ with mean $\boldsymbol{\mu} = \begin{bmatrix} \boldsymbol{\mu}_X \\ \boldsymbol{\mu}_Y \end{bmatrix}$ and covariance $\boldsymbol{\Sigma} = \begin{bmatrix} \boldsymbol{\Sigma}_{XX} & \boldsymbol{\Sigma}_{XY} \\ \boldsymbol{\Sigma}_{YX} & \boldsymbol{\Sigma}_{YY} \end{bmatrix}$.

The inverse of the partitioned covariance matrix $\boldsymbol{\Sigma}^{-1}$ can be written as:

$$\boldsymbol{\Sigma}^{-1} = \begin{bmatrix} \mathbf{M}_{XX} & \mathbf{M}_{XY} \\ \mathbf{M}_{YX} & \mathbf{M}_{YY} \end{bmatrix}$$

where:

*   $\mathbf{M}_{XX} = (\boldsymbol{\Sigma}_{XX} - \boldsymbol{\Sigma}_{XY}\boldsymbol{\Sigma}_{YY}^{-1}\boldsymbol{\Sigma}_{YX})^{-1} = \boldsymbol{\Sigma}_{X|Y}^{-1}$
*   $\mathbf{M}_{YY} = (\boldsymbol{\Sigma}_{YY} - \boldsymbol{\Sigma}_{YX}\boldsymbol{\Sigma}_{XX}^{-1}\boldsymbol{\Sigma}_{XY})^{-1} = \boldsymbol{\Sigma}_{Y|X}^{-1}$
*   $\mathbf{M}_{XY} = -\boldsymbol{\Sigma}_{X|Y}^{-1}\boldsymbol{\Sigma}_{XY}\boldsymbol{\Sigma}_{YY}^{-1}$
*   $\mathbf{M}_{YX} = -\boldsymbol{\Sigma}_{YY}^{-1}\boldsymbol{\Sigma}_{YX}\boldsymbol{\Sigma}_{X|Y}^{-1}$

From these identities, we can relate the terms. Specifically, we are interested in terms related to $\mathbf{X}$ given $\mathbf{Y}$.

The exponent of the joint density is $-\frac{1}{2} (\mathbf{z} - \boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{z} - \boldsymbol{\mu})$. Let $\tilde{\mathbf{x}} = \mathbf{x} - \boldsymbol{\mu}_X$ and $\tilde{\mathbf{y}} = \mathbf{y} - \boldsymbol{\mu}_Y$. The quadratic form becomes:

$$(\tilde{\mathbf{x}}^T \mathbf{M}_{XX} \tilde{\mathbf{x}} + \tilde{\mathbf{x}}^T \mathbf{M}_{XY} \tilde{\mathbf{y}} + \tilde{\mathbf{y}}^T \mathbf{M}_{YX} \tilde{\mathbf{x}} + \tilde{\mathbf{y}}^T \mathbf{M}_{YY} \tilde{\mathbf{y}})$$

We need to factor this expression to isolate terms dependent on $\mathbf{x}$ given $\mathbf{y}$.

The conditional mean $\boldsymbol{\mu}^*$ and conditional covariance $\boldsymbol{\Sigma}_{X|Y}$ are defined as:

*   **Conditional Mean:** $\boldsymbol{\mu}^* = E[\mathbf{X} \mid \mathbf{Y}=\mathbf{y}] = \boldsymbol{\mu}_X + \boldsymbol{\Sigma}_{XY}\boldsymbol{\Sigma}_{YY}^{-1}(\mathbf{y} - \boldsymbol{\mu}_Y)$
*   **Conditional Covariance:** $\boldsymbol{\Sigma}_{X|Y} = \boldsymbol{\Sigma}_{XX} - \boldsymbol{\Sigma}_{XY}\boldsymbol{\Sigma}_{YY}^{-1}\boldsymbol{\Sigma}_{YX}$

Note that the term $\mathbf{M}_{XX}^{-1}$ is precisely $\boldsymbol{\Sigma}_{X|Y}$.

The joint density can be rewritten (after some algebraic manipulation by completing the square in the exponent with respect to $\mathbf{x}$) as:

$$f_{\mathbf{X}, \mathbf{Y}}(\mathbf{x}, \mathbf{y}) = C \cdot \exp\left(-\frac{1}{2} (\mathbf{x} - \boldsymbol{\mu}^*)^T \boldsymbol{\Sigma}_{X|Y}^{-1} (\mathbf{x} - \boldsymbol{\mu}^*) - \frac{1}{2} (\mathbf{y} - \boldsymbol{\mu}_Y)^T \boldsymbol{\Sigma}_{YY}^{-1} (\mathbf{y} - \boldsymbol{\mu}_Y) \right)$$

where $C$ is a normalization constant that includes terms like $\frac{1}{\sqrt{(2\pi)^{n+m} |\boldsymbol{\Sigma}|}}$.

We also know that the marginal density $f_{\mathbf{Y}}(\mathbf{y})$ is:

$$f_{\mathbf{Y}}(\mathbf{y}) = \frac{1}{\sqrt{(2\pi)^m |\boldsymbol{\Sigma}_{YY}|}} \exp\left( -\frac{1}{2} (\mathbf{y} - \boldsymbol{\mu}_Y)^T \boldsymbol{\Sigma}_{YY}^{-1} (\mathbf{y} - \boldsymbol{\mu}_Y) \right)$$

Now, dividing $f_{\mathbf{X}, \mathbf{Y}}(\mathbf{x}, \mathbf{y})$ by $f_{\mathbf{Y}}(\mathbf{y})$:

$$f_{\mathbf{X}|\mathbf{Y}}(\mathbf{x} \mid \mathbf{y}) = \frac{C \cdot \exp\left(-\frac{1}{2} (\mathbf{x} - \boldsymbol{\mu}^*)^T \boldsymbol{\Sigma}_{X|Y}^{-1} (\mathbf{x} - \boldsymbol{\mu}^*) - \frac{1}{2} (\mathbf{y} - \boldsymbol{\mu}_Y)^T \boldsymbol{\Sigma}_{YY}^{-1} (\mathbf{y} - \boldsymbol{\mu}_Y) \right)}{\frac{1}{\sqrt{(2\pi)^m |\boldsymbol{\Sigma}_{YY}|}} \exp\left( -\frac{1}{2} (\mathbf{y} - \boldsymbol{\mu}_Y)^T \boldsymbol{\Sigma}_{YY}^{-1} (\mathbf{y} - \boldsymbol{\mu}_Y) \right)}$$

Notice that the term involving $(\mathbf{y} - \boldsymbol{\mu}_Y)^T \boldsymbol{\Sigma}_{YY}^{-1} (\mathbf{y} - \boldsymbol{\mu}_Y)$ cancels out. The remaining constant factors combine. It can be shown that $\frac{C}{\frac{1}{\sqrt{(2\pi)^m |\boldsymbol{\Sigma}_{YY}|}}} = \frac{1}{\sqrt{(2\pi)^n |\boldsymbol{\Sigma}_{X|Y}|}}$ using the determinant identity $|\boldsymbol{\Sigma}| = |\boldsymbol{\Sigma}_{YY}| |\boldsymbol{\Sigma}_{X|Y}|$.

Thus, we get:

$$f_{\mathbf{X}|\mathbf{Y}}(\mathbf{x} \mid \mathbf{y}) = \frac{1}{\sqrt{(2\pi)^n |\boldsymbol{\Sigma}_{X|Y}|}} \exp\left( -\frac{1}{2} (\mathbf{x} - \boldsymbol{\mu}^*)^T \boldsymbol{\Sigma}_{X|Y}^{-1} (\mathbf{x} - \boldsymbol{\mu}^*) \right)$$

This is the probability density function of a multivariate normal distribution with mean $\boldsymbol{\mu}^*$ and covariance matrix $\boldsymbol{\Sigma}_{X|Y}$, as required.

In summary, the derivation relies on:

1.  The definition of conditional probability ($f_{X|Y} = f_{X,Y} / f_Y$).
2.  The known forms of the joint and marginal Gaussian densities.
3.  Crucially, completing the square in the exponent of the joint density and using the Schur complement property for the inverse of a partitioned matrix to identify the conditional mean and covariance.""")

To show that the conditional distribution $f_{\mathbf{X}|\mathbf{Y}}(\mathbf{x} \mid \mathbf{y})$ for a jointly Gaussian random vector $\mathbf{Z} = egin{bmatrix} \mathbf{X} \ \mathbf{Y} \end{bmatrix}$ is also Gaussian, we start with the definition of conditional probability density:

$$f_{\mathbf{X}|\mathbf{Y}}(\mathbf{x} \mid \mathbf{y}) = rac{f_{\mathbf{X}, \mathbf{Y}}(\mathbf{x}, \mathbf{y})}{f_{\mathbf{Y}}(\mathbf{y})}$$

We know the joint density $f_{\mathbf{X}, \mathbf{Y}}(\mathbf{z})$ and the marginal density $f_{\mathbf{Y}}(\mathbf{y})$ (which is also Gaussian). The key is to complete the square in the exponent of the joint density function. 

Let $\mathbf{Z} = egin{bmatrix} \mathbf{X} \ \mathbf{Y} \end{bmatrix}$ with mean $oldsymbol{\mu} = egin{bmatrix} oldsymbol{\mu}_X \ oldsymbol{\mu}_Y \end{bmatrix}$ and covariance $oldsymbol{\Sigma} = egin{bmatrix} oldsymbol{\Sigma}_{XX} & oldsymbol{\Sigma}_{XY} \ oldsymbol{\Sigma}_{YX} & oldsymbol{\Sigma}_{YY} \end{bmatrix}$

<>:1: SyntaxWarning: invalid escape sequence '\m'
<>:1: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_4855/1744757572.py:1: SyntaxWarning: invalid escape sequence '\m'
  print("""To show that the conditional distribution $f_{\mathbf{X}|\mathbf{Y}}(\mathbf{x} \mid \mathbf{y})$ for a jointly Gaussian random vector $\mathbf{Z} = \begin{bmatrix} \mathbf{X} \\ \mathbf{Y} \end{bmatrix}$ is also Gaussian, we start with the definition of conditional probability density:


# Q3) Continuous Fault Monitoring of a Spindle Motor

A factory monitors the health of a high-speed CNC spindle. The critical variable is the **Internal Temperature ($X$)**, but because the motor is sealed, they can only observe the **Total Spindle Power ($Y$)**.

**The Setup:**
1.  **Hidden Temperature ($X$):** The standardized temperature deviation follows a standard normal distribution:
    $$X \sim \mathscr{N}(0, 1)$$
2.  **Mechanical Load ($Q$):** The torque required to cut material creates "noise" in the power reading. It is independent of temperature and follows:
    $$Q \sim \mathscr{N}(0, 4)$$
3.  **The Measurement ($Y$):** The power sensor $Y$ measures the combined effect of thermal load and mechanical torque:
$$Y = X + Q$$

---

**Answer the followings:**

**1. Joint Distribution**
Identify the distribution of the random varaible $(X, Y)$.
* Show that $Y \sim \mathscr{N}(0, 5)$.
* Find the covariance $\text{Cov}(X, Y)$.

**2. Information Geometry**
Explain why $X$ is **not** $\sigma(Y)$-measurable. In practical terms, if the power meter shows $Y = 5$, why can't the engineer know for certain if the motor is overheating ($X$ is high) or if the material being cut is simply very hard ($Q$ is high)?

**3. Conditional Expectation (The Best Estimate)**
Compute the conditional expectation $E[X \mid Y]$.
* Using the property for joint normals: $E[X \mid Y = y] = \mu_X + \frac{\text{Cov}(X,Y)}{\text{Var}(Y)}(y - \mu_Y)$.
* Express this as a "signal-to-total-variance" ratio.

**4. Verification of the Residual**
Let $\widehat{X} = E[X \mid Y]$. Define the estimation error (residual) as $Z = X - \widehat{X}$.
* Prove that $E[Z] = 0$.
* Verify that the error $Z$ is uncorrelated with the observation $Y$ (the Orthogonality Principle).

**5. Information Theory: Differential Entropy**
* Compute the differential entropy $h(X)$.
* Compute the conditional differential entropy $h(X \mid Y)$.
* *Note: The variance of the conditional distribution $(X \mid Y=y)$ is given by $\sigma_{X|Y}^2 = \sigma_X^2 - \frac{\text{Cov}(X,Y)^2}{\text{Var}(Y)}$.*

**6. Mutual Information**
Calculate $I(X; Y) = h(X) - h(X \mid Y)$.
* What percentage of the "uncertainty" about the temperature is removed by observing the power meter?
* How would this change if the mechanical noise $Q$ had a variance of $100$ instead of $4$?

**7. Diagnostic Interpretation**
The factory manager says: *"Since $Y = X + Q$, our best estimate of temperature should just be $Y$ itself."*
Using your result from Question 3, explain why $Y$ is a **biased** estimator of $X$ and why $E[X \mid Y]$ "shrinks" the measurement toward the mean (zero).



In [5]:
import numpy as np
from math import log, pi, sqrt

# Given parameters
mu_X = 0
var_X = 1
sigma_X = sqrt(var_X)

mu_Q = 0
var_Q = 4
sigma_Q = sqrt(var_Q)

# Since Q is independent of X, Cov(X, Q) = 0

print("--- Q3) Continuous Fault Monitoring of a Spindle Motor ---")

# --- 1. Joint Distribution ---
print("\n--- 1. Joint Distribution ---")

# Y = X + Q
# a) Show that Y ~ N(0, 5)
# Since X and Q are independent normal random variables, their sum is also normal.
# E[Y] = E[X + Q] = E[X] + E[Q] = mu_X + mu_Q = 0 + 0 = 0
# Var(Y) = Var(X + Q) = Var(X) + Var(Q) (due to independence) = var_X + var_Q = 1 + 4 = 5
mu_Y = mu_X + mu_Q
var_Y = var_X + var_Q
sigma_Y = sqrt(var_Y)

print(f"a) Distribution of Y:")
print(f"   E[Y] = E[X] + E[Q] = {mu_X} + {mu_Q} = {mu_Y}")
print(f"   Var(Y) = Var(X) + Var(Q) (due to independence) = {var_X} + {var_Q} = {var_Y}")
print(f"   Therefore, Y ~ N({mu_Y}, {var_Y}).")

# b) Find the covariance Cov(X, Y)
# Cov(X, Y) = Cov(X, X + Q)
#           = Cov(X, X) + Cov(X, Q)
#           = Var(X) + 0 (since X and Q are independent)
#           = var_X = 1
cov_XY = var_X

print(f"b) Covariance Cov(X, Y):")
print(f"   Cov(X, Y) = Cov(X, X + Q) = Var(X) + Cov(X, Q) = {var_X} + 0 = {cov_XY}")

print("\n--- 2. Information Geometry ---")
print("X is not \u03c3(Y)-measurable because Y is not a one-to-one function of X.")
print("Knowing Y (total power) doesn't uniquely determine X (internal temperature) because Q (mechanical load) is also a random variable contributing to Y.")
print("If the power meter shows Y=5, it could be due to X being high (overheating) and Q being low, or X being low and Q being high (hard material), or any combination in between.")
print("The engineer cannot know for certain if X is high because the 'noise' Q is also influencing the observation Y.")

# --- 3. Conditional Expectation (The Best Estimate) ---
print("\n--- 3. Conditional Expectation (The Best Estimate) ---")

# For jointly Gaussian random variables, E[X | Y=y] is given by:
# E[X | Y=y] = mu_X + (Cov(X,Y) / Var(Y)) * (y - mu_Y)

# Calculated values:
# mu_X = 0
# mu_Y = 0
# Cov(X,Y) = 1
# Var(Y) = 5

conditional_mean_coefficient = cov_XY / var_Y
E_X_given_Y_formula = f"E[X | Y=y] = {mu_X} + ({cov_XY} / {var_Y}) * (y - {mu_Y})"
E_X_given_Y_simplified = f"E[X | Y=y] = {conditional_mean_coefficient:.2f} * y"

print(f"a) Compute E[X | Y]:")
print(f"   E[X | Y=y] = \u03bc_X + (Cov(X,Y) / Var(Y)) * (y - \u03bc_Y)")
print(f"   E[X | Y=y] = {E_X_given_Y_formula}")
print(f"   E[X | Y=y] = {E_X_given_Y_simplified}")

print(f"b) Express as \"signal-to-total-variance\" ratio:")
print(f"   The coefficient (Cov(X,Y) / Var(Y)) can be written as Var(X) / (Var(X) + Var(Q))")
print(f"   = {var_X} / ({var_X} + {var_Q}) = {var_X} / {var_Y} = {conditional_mean_coefficient:.2f}")
print(f"   This ratio ({conditional_mean_coefficient:.2f}) represents the proportion of the total variance in Y that is attributed to the signal X.")

# --- 4. Verification of the Residual ---
print("\n--- 4. Verification of the Residual ---")

# Let X_hat = E[X | Y] = (1/5)Y
# Z = X - X_hat = X - (1/5)Y
# We know Y = X + Q, so Z = X - (1/5)(X + Q) = X - (1/5)X - (1/5)Q = (4/5)X - (1/5)Q

print("a) Prove that E[Z] = 0:")
E_Z = (4/5) * mu_X - (1/5) * mu_Q
print(f"   E[Z] = E[(4/5)X - (1/5)Q] = (4/5)E[X] - (1/5)E[Q] = (4/5)*{mu_X} - (1/5)*{mu_Q} = {E_Z}")
print(f"   Thus, E[Z] = 0.")

print("b) Verify that Z is uncorrelated with Y (Orthogonality Principle):")
# Cov(Z, Y) = Cov((4/5)X - (1/5)Q, X + Q)
#           = (4/5)Cov(X, X) + (4/5)Cov(X, Q) - (1/5)Cov(Q, X) - (1/5)Cov(Q, Q)
# Since X and Q are independent, Cov(X, Q) = Cov(Q, X) = 0.
# Cov(Z, Y) = (4/5)Var(X) - (1/5)Var(Q)
Cov_ZY = (4/5) * var_X - (1/5) * var_Q

print(f"   Cov(Z, Y) = (4/5)Var(X) - (1/5)Var(Q)")
print(f"             = (4/5)*{var_X} - (1/5)*{var_Q}")
print(f"             = {Cov_ZY}")
print(f"   Since Cov(Z, Y) = 0, Z and Y are uncorrelated.")

# --- 5. Information Theory: Differential Entropy ---
print("\n--- 5. Information Theory: Differential Entropy ---")

# a) Compute the differential entropy h(X)
# For a normal distribution N(mu, var), h(X) = 0.5 * log(2 * pi * e * var)
def differential_entropy_normal(variance):
    return 0.5 * log(2 * pi * np.exp(1) * variance)

h_X = differential_entropy_normal(var_X)
print(f"a) h(X) = 0.5 * log(2 * \u03c0 * e * Var(X))")
print(f"        = 0.5 * log(2 * \u03c0 * e * {var_X}) = {h_X:.4f} nats")

# b) Compute the conditional differential entropy h(X | Y)
# The variance of the conditional distribution (X | Y=y) is given by:
# sigma_X|Y^2 = sigma_X^2 - (Cov(X,Y)^2 / Var(Y))
var_X_given_Y = var_X - (cov_XY**2 / var_Y)
h_X_given_Y = differential_entropy_normal(var_X_given_Y)

print(f"b) h(X | Y):")
print(f"   Var(X | Y) = Var(X) - (Cov(X,Y)^2 / Var(Y))")
print(f"              = {var_X} - ({cov_XY}**2 / {var_Y}) = {var_X_given_Y:.2f}")
print(f"   h(X | Y) = 0.5 * log(2 * \u03c0 * e * Var(X | Y))")
print(f"            = 0.5 * log(2 * \u03c0 * e * {var_X_given_Y:.2f}) = {h_X_given_Y:.4f} nats")

# --- 6. Mutual Information ---
print("\n--- 6. Mutual Information ---")

I_X_Y = h_X - h_X_given_Y
print(f"a) I(X; Y) = h(X) - h(X | Y) = {h_X:.4f} - {h_X_given_Y:.4f} = {I_X_Y:.4f} nats")

percentage_removed = (I_X_Y / h_X) * 100
print(f"   Percentage of uncertainty about temperature removed by observing power meter: {percentage_removed:.2f}%")

# How would this change if the mechanical noise Q had a variance of 100 instead of 4?
var_Q_new = 100
var_Y_new = var_X + var_Q_new # 1 + 100 = 101
cov_XY_new = var_X # still 1

var_X_given_Y_new = var_X - (cov_XY_new**2 / var_Y_new)
h_X_given_Y_new = differential_entropy_normal(var_X_given_Y_new)
I_X_Y_new = h_X - h_X_given_Y_new
percentage_removed_new = (I_X_Y_new / h_X) * 100

print(f"\nb) If Var(Q) = {var_Q_new}:")
print(f"   New Var(Y) = {var_Y_new}")
print(f"   New Var(X | Y) = {var_X_given_Y_new:.4f}")
print(f"   New I(X; Y) = {I_X_Y_new:.4f} nats")
print(f"   New percentage of uncertainty removed: {percentage_removed_new:.2f}%")
print(f"   As mechanical noise (Var(Q)) increases, the mutual information decreases, meaning less uncertainty about X is removed by observing Y.")

# --- 7. Diagnostic Interpretation ---
print("\n--- 7. Diagnostic Interpretation ---")
print(f"The factory manager's suggestion to use Y as the estimator for X ($ \\widehat{{X}} = Y $) is biased.")
print(f"From part 3, we found E[X | Y=y] = {conditional_mean_coefficient:.2f} * y. If Y=y, then X_hat = {conditional_mean_coefficient:.2f}y.")
print(f"If the manager used Y as the estimate, then E[Y] = {mu_Y} and Var(Y) = {var_Y}.")
print(f"The bias of using Y as an estimator for X is E[Y] - E[X] = {mu_Y} - {mu_X} = {mu_Y - mu_X}.")
print(f"More importantly, E[X|Y] = {conditional_mean_coefficient:.2f}Y. Since {conditional_mean_coefficient:.2f} < 1, this shows a 'shrinkage' effect.")
print(f"When we observe Y, our best estimate for X is not Y itself, but a fraction of Y, scaled by the 'signal-to-total-variance' ratio.")
print(f"This shrinkage occurs because Y contains noise (Q) that is unrelated to X. To mitigate the effect of this noise, the conditional expectation pulls the estimate of X towards its prior mean (which is 0 in this standardized case) compared to simply using the observed Y value.")
print(f"For instance, if Y is observed as a large positive value, X is likely positive, but not as large as Y, because part of that large Y is probably due to a large Q. Therefore, we 'shrink' the estimate towards the mean (0) of X.")


--- Q3) Continuous Fault Monitoring of a Spindle Motor ---

--- 1. Joint Distribution ---
a) Distribution of Y:
   E[Y] = E[X] + E[Q] = 0 + 0 = 0
   Var(Y) = Var(X) + Var(Q) (due to independence) = 1 + 4 = 5
   Therefore, Y ~ N(0, 5).
b) Covariance Cov(X, Y):
   Cov(X, Y) = Cov(X, X + Q) = Var(X) + Cov(X, Q) = 1 + 0 = 1

--- 2. Information Geometry ---
X is not σ(Y)-measurable because Y is not a one-to-one function of X.
Knowing Y (total power) doesn't uniquely determine X (internal temperature) because Q (mechanical load) is also a random variable contributing to Y.
If the power meter shows Y=5, it could be due to X being high (overheating) and Q being low, or X being low and Q being high (hard material), or any combination in between.
The engineer cannot know for certain if X is high because the 'noise' Q is also influencing the observation Y.

--- 3. Conditional Expectation (The Best Estimate) ---
a) Compute E[X | Y]:
   E[X | Y=y] = μ_X + (Cov(X,Y) / Var(Y)) * (y - μ_Y)
   E[X | Y=